In [1]:
# Parameters
BATCH_MODE = "true"

# Complexity-03f — La chaîne des réductions : de 3SUM au triangle lopsided

**Série Complexity — accrétion de 03** · Public : **Recherche** · Hommage : **Josh Alman & Virginia Vassilevska Williams** (*Truly Subquadratic 3SUM and Truly Subcubic APSP via Triangles in Sparse Lopsided Graphs*, arXiv 2610.06783, 10/2026 — copie archivée au gisement `Bibliographie IA/Automata/Complexity Theory and Hardness`)

> « The web of reductions that was interpreted to imply hardness is exactly what turns a single new algorithm into faster algorithms for a myriad of problems at once. »
> — Alman & Vassilevska Williams, introduction (p. 5), notre traduction : *la toile de réductions qu'on interprétait comme une preuve de dureté est exactement ce qui transforme un seul nouvel algorithme en algorithmes plus rapides pour une myriade de problèmes d'un coup.*

## 0. Pourquoi ce carnet

Le [03e](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) a vérifié les cinq claims d'Alman & Vassilevska Williams **là où ils sont énoncés** : la machine word RAM, le témoin fermé, le contrôle quadratique. Il n'a pas montré **comment** un seul algorithme nouveau peut à la fois accélérer 3SUM, APSP, le produit (min,+), la k-Clique de poids nul et trois conjectures de matrices en ligne. La réponse tient en un mot : des **réductions** — celles-là mêmes qui, pendant quinze ans, servaient à prouver que ces problèmes étaient durs.

Ce carnet **marche la chaîne à pieds joints**, sur des instances :

1. les trois hypothèses de la complexité à grains fins et leur toile ;
2. la réduction de 3SUM au **triangle exact** (poids de somme nulle), construite et vérifiée sur corpus ;
3. le maillon **All-Edges Sparse Triangle** (la disjointness d'ensembles hors ligne), sa barrière $m^{4/3}$ et pourquoi les instances produites par les réductions semblaient la coller au mur ;
4. la version **lopsided** — un produit matriciel « fin » $X \in \mathbb{Z}^{N\times D}$, $Y \in \mathbb{Z}^{D\times N}$ avec $D \le N^{1/18}$ — implémentée à l'échelle jouet, entrées voulues contre produit complet ;
5. ce qui survit (SETH, Orthogonal Vectors, les modèles restreints) et ce qui tombe.

**Prérequis** : [03](Complexity-03-HierarchieTemporelle-Python.ipynb) (hiérarchie en temps), [02](Complexity-02-P-NP-Reduction-Python.ipynb) (l'idée de réduction), [03e](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) (les cinq claims et leur formalisation). Aucune ressource externe : tout est généré et mesuré ici.

In [2]:
import math
import random
import time
from fractions import Fraction

SEED = 20261111  # tout le carnet est rejouable a l'identique
rng = random.Random(SEED)
print(f"Imports OK : math, random, time, fractions — SEED={SEED}")

Imports OK : math, random, time, fractions — SEED=20261111


## 1. Trois hypothèses et une toile

La complexité à grains fins (*fine-grained complexity*) part d'un constat : pour une famille de problèmes polynomiaux, aucun progrès d'exposant depuis des décennies. Trois hypothèses ont servi de socle (papier, p. 3) :

| Hypothèse | Énoncé (word RAM, mots $O(\log n)$) | Meilleur connu avant 10/2026 |
|---|---|---|
| **3SUM** | $n$ entiers de taille polynomiale exigent $n^{2-o(1)}$ | $O(n^2)$, gagné seulement en polylog |
| **APSP** | plus courts chemins toutes paires exigent $n^{3-o(1)}$ | $O(n^3 / 2^{\Omega(\sqrt{\log n})})$ — sous-polynomial |
| **SETH** | CNF-SAT à $n$ variables, $O(n)$ clauses : pas de $(2-\varepsilon)^n$ | $2^n$ (à $c^n$ près) |

L'outil du domaine est la **réduction à grains fins** : de $A$ vers $B$ relativement à $a(n)$ et $b(n)$ — si $B$ se résout en $O(b(n)^{1-\varepsilon})$, alors $A$ se résout en $O(a(n)^{1-\delta})$. Une réduction prouve la dureté de $A$ à partir de celle de $B$… et devient un **algorithme pour $A$** le jour où $B$ devient facile.

```mermaid
flowchart LR
    SETH["SETH — CNF-SAT<br/>(intacte)"]
    OV["Orthogonal Vectors<br/>(intacte)"]
    TSUM["Classe 3SUM<br/>(geom., conv-3SUM...)"]
    APSPC["Classe APSP<br/>((min,+), Metricity,<br/>Tree Edit Distance...)"]
    KCL["k-Clique de poids nul"]
    ET["Triangle Exact"]
    AEST["All-Edges Sparse Triangle<br/>(barrière m^4/3)"]
    LOP["Lopsided All-Edges<br/>Sparse Triangle<br/>— le moteur du papier"]
    SETH -.->|non réduit| OV
    TSUM --> ET
    APSPC --> ET
    KCL --> ET
    ET --> AEST
    AEST --> LOP
    LOP -.->|nouvel algorithme| TSUM
    LOP -.->|nouvel algorithme| APSPC
    LOP -.->|nouvel algorithme| KCL
```

La flèche pleine descend vers le problème central ; la flèche pointillée est le renversement d'octobre 2026 : le moteur lopsided remonte la toile. SETH et Orthogonal Vectors ne figurent **pas** dans la toile descendante du papier — elles restent intactes.

In [3]:
def trois_sommes_quadratique(nombres):
    """Decide 3SUM (trois indices distincts, somme nulle) en O(n^2).
    Retourne (trouve, triple, operations)."""
    s = sorted(nombres)
    n = len(s)
    ops = 0
    for i in range(n - 2):
        cible = -s[i]
        lo, hi = i + 1, n - 1
        while lo < hi:
            ops += 1
            deux = s[lo] + s[hi]
            if deux == cible:
                return True, (s[i], s[lo], s[hi]), ops
            if deux < cible:
                lo += 1
            else:
                hi -= 1
    return False, None, ops


def instance_3sum(n, avec_solution, borne, rng):
    """Instance 3SUM a n entiers dans [-borne, borne]."""
    vals = rng.sample(range(-borne, borne + 1), n)
    if avec_solution:
        a, b = rng.sample(range(1, borne // 2), 2)
        vals[0], vals[1], vals[2] = a, b, -(a + b)
    else:
        # corpus negatif : on force un trou autour de zero parite (heuristique,
        # l'exhaustivite n'est pas requise : le controle est fait par oracle lent)
        pass
    return vals


def oracle_3sum_lent(nombres):
    """Oracle O(n^3) : triple distinct de somme nulle."""
    n = len(nombres)
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nombres[i] + nombres[j] + nombres[k] == 0:
                    return True, (nombres[i], nombres[j], nombres[k])
    return False, None


n_demo = 60
inst_oui = instance_3sum(n_demo, True, 500, rng)
inst_non = [3 * k + 1 for k in range(n_demo)]  # tous 1 mod 3 et > 0 : somme de trois ne fait jamais 0
assert not oracle_3sum_lent(inst_non)[0]

t_oui, tri_oui, ops_oui = trois_sommes_quadratique(inst_oui)
t_non, _, ops_non = trois_sommes_quadratique(inst_non)
print(f"Instance AVEC solution : trouve={t_oui}, triple={tri_oui}, ops={ops_oui}")
print(f"Instance SANS solution (oracle lent d'accord) : trouve={t_non}, ops={ops_non}")
print(f"Rapport ops_non / (n^2/2) = {ops_non / (n_demo ** 2 / 2):.3f} — le cadrage double balayage")

Instance AVEC solution : trouve=True, triple=(-447, 127, 320), ops=101
Instance SANS solution (oracle lent d'accord) : trouve=False, ops=1711
Rapport ops_non / (n^2/2) = 0.951 — le cadrage double balayage


### Lecture du résultat — le mur quadratique, mesuré

L'instance **sans** solution coûte $n^2/2$ opérations de sommet à sommet (le double balayage épuise chaque paire) : c'est exactement ce que l'hypothèse 3SUM postule incontournable — et exactement ce que le nouveau $O(n^{1.9992})$_casse d'un cheveu ($n^{0{,}0008}$, cf. [03e](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb)).

## 2. De 3SUM au triangle exact

**Triangle Exact** : un graphe tripartite à poids entiers sur les arêtes ; existe-t-il un triangle dont les trois poids somment à zéro ?

La construction (forme linéaire, maillon classique de la chaîne — [VW10, VW18] dans le papier, p. 4) part du 3SUM **triparti** : trois ensembles $A, B, C$, cherche-t-on $a+b+c=0$ à travers les trois ? Alors le graphe a trois parties copiant $A, B, C$, et les poids :

$$w(u^A_a, u^B_b) = a, \qquad w(u^B_b, u^C_c) = b, \qquad w(u^C_c, u^A_a) = c$$

Chaque triangle $(a,b,c)$ porte la somme $a+b+c$ : le triangle de poids nulle somme **est** le triple nul. Le graphe est complet tripartite — le triangle exact parcourt tous les triples, y compris ceux qui réutilisent la même *valeur* : c'est pourquoi on part du 3SUM triparti (indices disjoints par construction).

Le pont depuis le 3SUM mono-ensemble (calcul du carnet, pont pédagogique) : colorier les $n$ éléments en trois couleurs $A, B, C$ ; un triple nul d'indices distincts est bien colorié avec probabilité $6/27 = 2/9$ (chaque élément reçoit l'une des 3 couleurs indépendamment ; les 6 permutations bijectives conviennent). $r$ coloriages indépendants ratent le triple avec probabilité $(7/9)^r$.

In [4]:
def graphe_depuis_3sum_triparti(A, B, C):
    """Triangle Exact complet tripartite : w(AB)=a, w(BC)=b, w(CA)=c."""
    return {"A": list(A), "B": list(B), "C": list(C)}


def somme_triangle(g, ia, ib, ic):
    a, b, c = g["A"][ia], g["B"][ib], g["C"][ic]
    return a + b + c  # w(AB)=a, w(BC)=b, w(CA)=c


def triangle_exact_brut(g):
    """Oracle brutal O(|A|.|B|.|C|) : premier triangle de somme nulle."""
    for ia in range(len(g["A"])):
        for ib in range(len(g["B"])):
            for ic in range(len(g["C"])):
                if somme_triangle(g, ia, ib, ic) == 0:
                    return True, (g["A"][ia], g["B"][ib], g["C"][ic])
    return False, None


def triparti_par_coloriage(nombres, rng):
    """3SUM mono-ensemble -> 3SUM triparti par 3-coloriage."""
    A, B, C = [], [], []
    for v in nombres:
        c = rng.randrange(3)
        (A if c == 0 else B if c == 1 else C).append(v)
    return A, B, C


# --- verification d'equivalence sur corpus ---
# cote SANS triple (valeurs 1 mod 3 positives) : aucun coloriage ne doit produire
# de triangle de somme nulle ; cote AVEC triple : 30 coloriages doivent le capturer.
accords, fantomes, captures_oui = 0, 0, 0
for essai in range(40):
    n = rng.randrange(12, 22)
    if essai % 2 == 0:
        vals = instance_3sum(n, True, 120, rng)
        oracle = oracle_3sum_lent(vals)[0]
    else:
        vals = [3 * k + 1 for k in range(n)]
        oracle = False
    assert oracle == (essai % 2 == 0)
    vu = False
    for r in range(30):
        A, B, C = triparti_par_coloriage(vals, rng)
        if not (A and B and C):
            continue
        if triangle_exact_brut(graphe_depuis_3sum_triparti(A, B, C))[0]:
            vu = True
            break
    attendu = oracle
    if vu == attendu:
        accords += 1
    if vu and not attendu:
        fantomes += 1
    if attendu and vu:
        captures_oui += 1

print(f"Corpus 40 instances (20 avec / 20 sans) : {accords} accords, {fantomes} triangle fantome")
print(f"Cote 'avec triple' captures en <=30 coloriages : {captures_oui}/20")

Corpus 40 instances (20 avec / 20 sans) : 40 accords, 0 triangle fantome
Cote 'avec triple' captures en <=30 coloriages : 20/20


In [5]:
# Mesure du 2/9 : combien de coloriages captent le triple ?
rng2 = random.Random(SEED + 1)
captures = 0
essais = 300
r_coloriages = 5
for _ in range(essais):
    n = 15
    vals = instance_3sum(n, True, 120, rng2)
    vu = False
    for _ in range(r_coloriages):
        A, B, C = triparti_par_coloriage(vals, rng2)
        if A and B and C and triangle_exact_brut(graphe_depuis_3sum_triparti(A, B, C))[0]:
            vu = True
            break
    captures += vu
pr = 1 - Fraction(7, 9) ** r_coloriages
print(f"Captures en {r_coloriages} coloriages : {captures}/{essais} = {captures/essais:.3f}")
print(f"Theorie 1-(7/9)^{r_coloriages} = {float(pr):.3f} (le 2/9 par coloriage est la borne exacte du triple fixe)")

Captures en 5 coloriages : 267/300 = 0.890
Theorie 1-(7/9)^5 = 0.715 (le 2/9 par coloriage est la borne exacte du triple fixe)


### Lecture du résultat — le pont tient par 2/9

Un coloriage capture un triple nul **fixé** avec probabilité $2/9 \approx 0{,}222$ ; cinq coloriages indépendants le capturent à $1-(7/9)^5 \approx 0{,}715$. Le carnet mesure $0{,}890$ — **au-dessus** de la borne du triple fixé, parce qu'une instance de $15$ valeurs dans $[-120, 120]$ porte souvent plusieurs triples nuls, et que le coloriage n'a besoin d'en capturer **aucun en particulier** : l'événement « capturer au moins un triple » contient celui de capturer un triple fixé. Sa probabilité est donc au moins $1-(7/9)^5$ ; cette formule est exacte pour le triple fixé. La fréquence $267/300$ est une estimation empirique, pas une borne rigoureuse sur ces probabilités. Le papier, lui, n'a pas besoin de ce pont aléatoire : ses réductions publiées ([Pǎ10, KPP16] cités p. 6) vont directement de 3SUM au **lopsided**, et sont **dérandomisées** (p. 6) — le coloriage aléatoire n'est qu'un jouet pour voir le maillon.

## 3. La mèche courte : All-Edges Sparse Triangle

**All-Edges Sparse Triangle** : un graphe creux à $m$ arêtes ; pour **chaque** arête, décide-t-elle dans un triangle ? C'est la disjointness d'ensembles **hors ligne** (papier, p. 4) : chaque sommet porte un ensemble, chaque arête pose la question « vos ensembles s'intersectent-ils ? ».

Deux bornes classiques encadraient le problème :

- **énumérer les triangles** coûte $O(m^{3/2})$ [IR78, CN85] ;
- le partage haut-degré/bas-degré [AYZ97] gagne sur les graphes denses : $O(m^{2\omega/(\omega+1)})$ — **$m^{4/3}$ même si $\omega = 2$**.

Et le piège : les instances que produisent les réductions (p. 4) ont des sommets de degré $\approx \sqrt{n}$ — sur elles, $m^{4/3} = \Theta(n^2)$ pour $\omega=2$. La borne d'énumération $m^{3/2}$ est plus grande : elle ne se confond pas avec celle du partage. Tout laissait croire le mur infranchissable : le produit matriciel rapide semblait n'être bon que pour les entrées denses, or on ne demande qu'un ensemble **creux** de sorties.

In [6]:
def toutes_aretes_triangle(graphe_adj):
    """All-Edges Sparse Triangle par appartenance : pour chaque arete (u,v),
    u et v ont-ils un voisin commun ? (disjointness hors ligne.)
    graphe_adj : dict sommet -> set de voisins. Retourne dict arete -> bool."""
    res = {}
    vues = set()
    for u, vois in graphe_adj.items():
        for v in vois:
            arete = (min(u, v), max(u, v))
            if arete in vues:
                continue
            vues.add(arete)
            res[arete] = len(vois & graphe_adj[v]) > 0
    return res


def evalue_graphe_lopsided(N, D, densite, rng):
    """Graphe tripartite lopsided : parties P1, P2 de N sommets,
    partie moyenne M de D sommets. Adjacence P-M creuse."""
    adj = {}
    for p in range(2 * N):
        adj[p] = set()
    for m in range(2 * N, 2 * N + D):
        adj[m] = set()
    for p in range(2 * N):
        for m in range(2 * N, 2 * N + D):
            if rng.random() < densite:
                adj[p].add(m)
                adj[m].add(p)
    return adj


# sur le lopsided, l'arete P-M est dans un triangle ssi le sommet P partage
# un autre voisin M' avec un sommet P'... on mesure via les paires (P1,P2):
def paires_avec_voisin_commun(adj, N, D):
    """Pour chaque paire (p1, p2) des parties externes : nb de voisins communs
    dans la partie moyenne. C'est exactement l'entree (XY)[p1,p2] du papier."""
    milieux = set(range(2 * N, 2 * N + D))
    inter = {}
    for p1 in range(N):
        for p2 in range(N, 2 * N):
            inter[(p1, p2)] = len(adj[p1] & adj[p2] & milieux)
    return inter


N_jouet, D_jouet, dens = 40, 6, 0.55
adj = evalue_graphe_lopsided(N_jouet, D_jouet, dens, rng)
t0 = time.perf_counter()
inter = paires_avec_voisin_commun(adj, N_jouet, D_jouet)
t1 = time.perf_counter()
nb_pos = sum(1 for v in inter.values() if v > 0)
print(f"Graphe lopsided N={N_jouet}, D={D_jouet}, densite={dens}")
print(f"Paires externes = {len(inter)} = N^2 ; paires avec voisin commun : {nb_pos}")
print(f"Temps produit-complet des intersections : {(t1 - t0)*1000:.2f} ms")

Graphe lopsided N=40, D=6, densite=0.55
Paires externes = 1600 = N^2 ; paires avec voisin commun : 1449
Temps produit-complet des intersections : 0.86 ms


### Lecture du résultat — $m^{4/3}$ sur le graphe de réduction

Pour $n$ sommets de degré moyen $\sqrt{n}$, la somme des degrés vaut $2m$, donc $m = n^{3/2}/2$. Les deux bornes donnent alors :

$$m^{3/2} = n^{9/4}/2^{3/2}, \qquad m^{4/3} = n^2/2^{4/3}.$$

L'énumération a donc un exposant $9/4$ : elle dépasse le mur quadratique asymptotiquement. Le partage AYZ, lui, donne $\Theta(n^2)$ lorsque $\omega=2$ ; les constantes ne changent pas cet ordre. C'est ce verrou que le papier fait sauter pour les instances **lopsided**.

## 4. La version lopsided et le produit matriciel fin

Le moteur (papier, p. 5-6) : soit $X \in \mathbb{Z}^{N\times D}$ et $Y \in \mathbb{Z}^{D\times N}$ avec $D \le N^{1/18}$ (un produit « **fin** »), et un ensemble $W$ d'au plus $N^2/\sqrt{D}$ positions « **voulues** ». Alors les entrées $(XY)[I,J]$ pour $(I,J) \in W$ se calculent en

$$O\left(N^2 / D^{0{,}063}\right) \text{ opérations}$$

— polynômialement **moins** que le temps d'écrire $XY$ entier ($\Omega(N^2)$) ou de calculer $|W|$ produits scalaires un à un. Lue en graphe : $X$ et $Y$ sont les biadjacences vers la partie moyenne, $(XY)[a,b]$ **compte** les milieux communs — la version comptage de Lopsided All-Edges Sparse Triangle, qui décide la détection.

Sur un graphe tripartite à deux parties de $n$ sommets et une partie moyenne de $n^{\varepsilon}$ : compter les triangles à travers $|W|$ paires prescrites en $O(|W|\,n^{0{,}437\varepsilon} + n^{2-0{,}063\varepsilon})$ pour $\varepsilon < 1/18$ — vraiment sous-quadratique pour tout $\varepsilon < 0{,}1204$.

In [7]:
def produit_complet_ops(N, D, W=None):
    """Nombre de produits scalaires du produit COMPLET : N^2 produits de longueur D."""
    return N * N * D


def produit_voulu_ops(N, D, taille_W):
    """Produits scalaires du calcul NAIF voulu : |W| produits de longueur D.
    (Le papier fait mieux : O(N^2 / D^0.063) — compare les deux regimes.)"""
    return taille_W * D


# Comparaison de formules pour N croissant ; voir le domaine admissible ci-dessous
print(f"{'N':>5} {'D=N^e':>7} {'|W|<=N^2/sqrt(D)':>17} {'naif |W|.D':>11} {'complet N^2.D':>13} {'N^2/D^0.063':>12}")
for N in (64, 128, 256, 512, 1024):
    D = max(2, round(N ** (1 / 18)))  # ici D=2, hors domaine pour ces N
    borne_W = int(N * N / math.sqrt(D))  # plafond autorise sur |W|
    print(f"{N:>5} {D:>7} {borne_W:>17} {produit_voulu_ops(N, D, borne_W):>11} "
          f"{produit_complet_ops(N, D):>13} {N*N / D**0.063:>12.0f}")

    N   D=N^e  |W|<=N^2/sqrt(D)  naif |W|.D complet N^2.D  N^2/D^0.063
   64       2              2896        5792          8192         3921
  128       2             11585       23170         32768        15684
  256       2             46340       92680        131072        62736
  512       2            185363      370726        524288       250943
 1024       2            741455     1482910       2097152      1003772


### Lecture du résultat — trois régimes, un seul gagnant

Dans le domaine $D \le N^{1/18}$, prenons $|W|$ au plafond $N^2/\sqrt{D}$. Le calcul naïf voulu coûte $|W|D=N^2D^{1/2}$, tandis que le produit complet coûte $N^2D$. Le rapport entre le premier et l'expression de la borne est

$$\frac{N^2D^{1/2}}{N^2/D^{0{,}063}}=D^{0{,}563}.$$

C'est une comparaison d'ordres asymptotiques : la constante cachée dans $O(\cdot)$ empêche d'en déduire un facteur de temps effectif pour une instance finie. Pour $D=N^{1/18}$, ce rapport croît polynomialement en $N$. Le cœur technique modifie l'algorithme rectangulaire de Coppersmith (identité à dix multiplications de Schönhage, p. 6) pour ne calculer que les entrées voulues.

Le tableau est une comparaison **extrapolée hors du domaine** : toutes ses lignes ont $D=2$ et $N<2^{18}$, donc $2>N^{1/18}$. Pour $D=2$, l'hypothèse exige $N\ge 2^{18}$ ; les nombres du tableau ne démontrent donc pas l'application du théorème à ces instances. L'arrondi de $|W|$ explique les petits écarts aux expressions continues.

À $D=2$, ces expressions valent environ $1{,}414N^2$ pour le calcul naïf et $0{,}957N^2$ pour la borne, soit un rapport d'environ $1{,}48$. Il dépasse $2$ lorsque $D>2^{1/0{,}563}\approx 3{,}43$. À $D=5$, la condition du théorème demande $N\ge 5^{18}$, soit plus de $10^{12}$. Comme dans le [03e](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb), ces comparaisons d'exposants ne sont pas des chronomètres.


## 5. Ce qui survit, ce qui tombe

Le verdict d'octobre 2026 (Théorème 2, p. 7 ; OMv Théorème 4, p. 8) :

| Objet | Statut après le papier |
|---|---|
| Hypothèse 3SUM | **réfutée** — déterministe $O(n^{1{,}9992})$ word RAM ; $O(n^{1{,}998})$ Las Vegas en RAM réelle |
| Hypothèse APSP | **réfutée** — $O(n^{2{,}9995})$ (produit (min,+) idem) |
| Triangle Exact | $O(n^{2{,}9983})$ déterministe ; $O(n^{2{,}998})$ Las Vegas |
| k-Clique de poids nul | $O(n^{k-\varepsilon})$ pour un $\varepsilon > 0$ |
| Conjectures OMv « à indice fin » | **réfutées** pour $\tau < 1/18$ (précalcul $O(n^{2-0{,}063\tau})$) |
| Bornes dynamiques principales (inverse matriciel dynamique) | **intactes** — elles utilisent $\tau \approx 0{,}53$, loin du régime fin |
| **SETH**, Orthogonal Vectors | **intactes** — pas dans la toile du papier |
| Arbres de décision 3-linéaires pour 3SUM : $\Omega(n^2)$ | **intactes** [Eri99, AC05] — le nouvel algorithme sort du modèle |
| Programmes droits sur le semi-anneau (min,+) : $n^3$ | **intacts** [Ker70] — sortie du modèle par hachage des poids |

La sortie des modèles restreints (p. 7) : les réductions connues **retirent les poids** (hachage, ou l'astuce de Fredman pour les entrées réelles), puis comptent des triangles de graphe **non pondéré** par produit matriciel entier. Le comptage est plus fort que la détection — et la classe APSP entière suit par la toile.

In [8]:
verdicts = [
    ("3SUM", "n^1.9992 det / n^1.998 LV", "REFUTEE"),
    ("APSP", "n^2.9995 det / n^2.998 LV", "REFUTEE"),
    ("Triangle Exact", "n^2.9983 det", "REFUTEE (a sa dureté conditionnelle)"),
    ("(min,+)-produit", "n^2.9995 det", "REFUTEE"),
    ("k-Clique poids nul", "n^(k-eps)", "REFUTEE"),
    ("OMv indice fin (tau<1/18)", "precalc n^2-0.063tau", "REFUTEE"),
    ("SETH", "—", "INTACTE"),
    ("Orthogonal Vectors", "—", "INTACTE"),
    ("3-LDT 3SUM Omega(n^2)", "hors du modele", "INTACTE (modele restreint)"),
    ("SLP (min,+) n^3", "hors du modele", "INTACTE (modele restreint)"),
]
print(f"{'Objet':<28} {'Algo / statut':<26} Verdict")
print("-" * 78)
for obj, algo, verdict in verdicts:
    print(f"{obj:<28} {algo:<26} {verdict}")

Objet                        Algo / statut              Verdict
------------------------------------------------------------------------------
3SUM                         n^1.9992 det / n^1.998 LV  REFUTEE
APSP                         n^2.9995 det / n^2.998 LV  REFUTEE
Triangle Exact               n^2.9983 det               REFUTEE (a sa dureté conditionnelle)
(min,+)-produit              n^2.9995 det               REFUTEE
k-Clique poids nul           n^(k-eps)                  REFUTEE
OMv indice fin (tau<1/18)    precalc n^2-0.063tau       REFUTEE
SETH                         —                          INTACTE
Orthogonal Vectors           —                          INTACTE
3-LDT 3SUM Omega(n^2)        hors du modele             INTACTE (modele restreint)
SLP (min,+) n^3              hors du modele             INTACTE (modele restreint)


## 6. Exercices

Trois exercices, à compléter sans casser l'exécution (les stubs retournent `None` ; le carnet reste exécutable de bout en bout).

### Exercice 1 — le côté négatif du pont

Le corpus « sans solution » de la section 2 était contrôlé par l'oracle lent. Étendez la vérification : générez 10 instances **sans** triple nul (oracle lent d'accord), passez-les par 30 coloriages chacune, et vérifiez qu'**aucune** ne produit de triangle de somme nulle — le coloriage ne peut pas inventer un triple. Retournez le nombre d'instances testées et le nombre de triangles fantômes (attendu : 0).

In [9]:
def exercice1_sans_solution(nb_instances=10, nb_coloriages=30):
    """Retourne (nb_testees, nb_triangles_fantomes) -- attendu (10, 0).
    TODO etudiant"""
    # Etape 1 : generer nb_instances instances SANS triple nul (oracle_3sum_lent)
    # Etape 2 : pour chaque, nb_coloriages coloriages -> graphe triparti
    # Etape 3 : compter les triangles de somme nulle (triangle_exact_brut)
    # Indice : un graphe complet tripartite depuis des valeurs sans triple nul
    #          ne peut pas produire de somme nulle -- prouvez-le par la mesure.
    print("Exercice 1 a completer")
    return None  # TODO etudiant


r1 = exercice1_sans_solution()
print(f"Exercice 1 : {r1}")

Exercice 1 a completer
Exercice 1 : None


### Exercice 2 — All-Edges Sparse Triangle par énumération

Implémentez le comptage **par arête** sur le graphe lopsided de la section 3 : pour chaque arête $(u,v)$, le nombre de triangles qui la contiennent (pas seulement oui/non). Combien d'arêtes sont dans au moins un triangle ?

In [10]:
def exercice2_triangles_par_arete(adj):
    """Retourne dict arete -> nb de triangles la contenant.
    TODO etudiant"""
    # Etape 1 : pour chaque arete (u,v), intersector les voisinages
    # Etape 2 : chaque voisin commun w forme le triangle (u,v,w)
    # Indice : l'arete P1-P2 n'existe pas dans le biparti -- seules les aretes
    #          vers la partie moyenne (et d'eventuelles aretes P-P) comptent.
    print("Exercice 2 a completer")
    return None  # TODO etudiant


r2 = exercice2_triangles_par_arete(adj)
print(f"Exercice 2 : {r2}")

Exercice 2 a completer
Exercice 2 : None


### Exercice 3 — le régime du wanted set

Pour $\varepsilon \in \{1/36, 1/18\}$ et $N \in \{2^6, 2^8, 2^{10}\}$, posez formellement $D=N^\varepsilon$ et calculez $|W|_{\max}/N^2$, puis le rapport entre le coût naïf $|W|_{\max}D$ et l'expression $N^2/D^{0{,}063}$. À partir de quel $D$ ce rapport dépasse-t-il 2 ?

Pour interpréter les résultats, distinguez cette comparaison de formules d'un temps mesuré. $D$ doit être entier pour une dimension matricielle : discutez l'effet d'un arrondi et vérifiez à nouveau $D \le N^{1/18}$. Le plafond sur $|W|$ est une hypothèse du résultat cité ; il ne dit pas à lui seul ce qui se passe au-dessus.

In [11]:
def exercice3_regime_wanted():
    """Retourne la liste [(eps, N, rapport_W_sur_N2, gain_borne), ...].
    TODO etudiant"""
    # Etape 1 : D = N^eps ; W_max = N^2 / sqrt(D)
    # Etape 2 : naif = W_max * D ; borne = N^2 / D^0.063 ; gain = naif / borne
    # Indice : gain > 2 des que D^0.563 > 2 -- dans quel regime de eps ?
    print("Exercice 3 a completer")
    return None  # TODO etudiant


r3 = exercice3_regime_wanted()
print(f"Exercice 3 : {r3}")

Exercice 3 a completer
Exercice 3 : None


## Conclusion

| Maillon | Ce qu'on a fait | Mesure |
|---|---|---|
| Mur 3SUM | double balayage sur instance sans solution | $\approx n^2/2$ opérations |
| Pont coloriage | 3SUM mono-ensemble → triparti | $2/9$ par coloriage, $1-(7/9)^5$ à cinq essais |
| Réduction 3SUM → Triangle Exact | formes linéaires $w(AB)=a$ etc. | 40/40 accords sur corpus |
| Barrière $m^{4/3}$ | AYZ97 = mur sur instances de réduction (degré $\sqrt{n}$) | ordre de grandeur comparé |
| Moteur lopsided | produit fin, entrées voulues vs complet | $|W| \cdot D$ vs $N^2 D$ vs $N^2/D^{0{,}063}$ |
| Toile après octobre 2026 | 10 verdicts déroulés | 6 réfutées, 4 intactes |

## À retenir

1. **Une réduction est un certificat à double emploi.** Celle qui prouvait « 3SUM est dur si Triangle Exact est dur » devient « 3SUM est rapide dès que Triangle Exact est rapide » — la toile de dureté est le conduit de l'accélération (p. 5 : *la toile interprétée comme dureté est exactement ce qui transforme un algorithme en une myriade*).
2. **La construction est des formes linéaires.** $w(u^A_a, u^B_b) = a$, $w(u^B_b, u^C_c) = b$, $w(u^C_c, u^A_a) = c$ : la somme du triangle est $a+b+c$, le triangle nul **est** le triple nul — vérifié sur corpus, 0 fantôme sur le côté négatif.
3. **Le verrou était structurel, pas numérique.** Sur les instances de réduction (degré $\approx\sqrt{n}$), la borne d’énumération $m^{3/2}$ est en $\Theta(n^{9/4})$, tandis que le partage AYZ $m^{4/3}$ est en $\Theta(n^2)$ pour $\omega=2$ : la sortie creuse semblait hors de portée du produit matriciel. Le moteur fin ($D \le N^{1/18}$, $|W| \le N^2/\sqrt{D}$, $O(N^2/D^{0{,}063})$) calcule **moins** que ce qu'il faudrait pour écrire le produit.
4. **Réfuter une hypothèse ne refute pas le domaine.** SETH et Orthogonal Vectors restent debout, les bornes en modèles restreints (3-linéaires, programmes droits, comparaison de chemins) restent vraies — le nouvel algorithme **sort** de ces modèles (hachage des poids, comptage entier sur graphes non pondérés).

## Vérifiez votre compréhension

1. Pourquoi l'instance **sans** solution coûte-t-elle $n^2/2$ opérations au double balayage, et pas l'instance **avec** ?
2. Le coloriage aléatoire capture un triple nul donné avec probabilité $2/9$ — d'où sortent le 6 et le 27 du calcul ?
3. Sur une instance de réduction à degré $\approx\sqrt{n}$, montrez que $m^{4/3} \approx n^2$ (combien d'arêtes $m$ a-t-elle ?).
4. À $D = N^{1/18}$, pourquoi $N^2/D^{0{,}063}$ est-il **polynômialement** en dessous de $|W| \cdot D$ pris à $|W| = N^2/\sqrt{D}$ — et pourquoi le tableau à $N = 1024$ ne permet-il pas de conclure à un gain mesuré ?

## Pour aller plus loin

- **Carnet jumeau** : [Complexity-03e — barrières d'exposants, word RAM](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) — les cinq claims vérifiés à la machine, le témoin fermé, le facteur $n^{0{,}0008}$ calculé.
- **L'autre annonce vérifiée** : [Complexity-03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) — pourquoi elle, en revanche, attend toujours sa preuve.
- **Le panorama** : [Complexity-07c — le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) — où s'inscrit le contrepoint Williams–Alman du 5 octobre.
- **Carnet suivant de la série** : [Complexity-04 — algorithmes en ligne](Complexity-04-OnlineAlgorithms-Python.ipynb) — quitter la toile d'hypothèses pour les décisions sans connaître la suite.
- **Source** : Josh Alman & Virginia Vassilevska Williams, *Truly Subquadratic 3SUM and Truly Subcubic APSP via Triangles in Sparse Lopsided Graphs*, arXiv [2610.06783](https://arxiv.org/abs/2610.06783) (10/2026, 76 p.) — copie archivée au gisement `Bibliographie IA/Automata/Complexity Theory and Hardness`. Toute attribution du carnet se lit contre cette copie.